# TFG — Análisis automático de jugadas de voleibol con visión por computador

**Alumno:** Alejandro Márquez Malia  
**Tutor:** Jamal Toutouh El Alamin  
**Universidad:** Universidad de Málaga  

---

## Estructura del notebook

| Módulo | Celdas | Descripción |
|--------|--------|-------------|
| 0. Setup | 1-3 | Instalación, imports, Drive y rutas |
| 1. Calibración | 4-6 | Homografía pixel→metros |
| 2. Tracking | 7-8 | YOLOv8s + ByteTrack |
| 3. Limpieza | 9-11 | Filtrado, re-tracking, fusión de IDs |
| 4. Análisis espacial | 12-14 | Clustering K-means, métricas, dashboard |
| 5. Carga física | 15-16 | Zonas Z1-Z5 estilo medicina deportiva |
| 6. Estimación de pose | 17-20 | YOLOv8s-pose, ángulos, clasificación de acciones |
| 7. Evaluación | 21 | Métricas del tracker |
| 8. Vídeo anotado | 22 | Exportación con anotaciones |

> **Nota:** Ejecutar las celdas en orden. Los módulos 5-8 dependen de los CSVs generados en el módulo 4.


## Módulo 0 — Setup


### Celda 1 — Instalación de dependencias
> Ejecutar **una sola vez** por sesión de Colab.


In [ ]:
# Celda 1 — Instalación de dependencias
!pip install ultralytics --quiet
!pip install lap --quiet          # requerido por ByteTrack
!pip install supervision --quiet  # anotación de vídeo

print("✅ Librerías instaladas")

### Celda 2 — Importaciones


In [ ]:
# Celda 2 — Importaciones
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from collections import Counter
from scipy.spatial import ConvexHull
from scipy.stats import gaussian_kde
from sklearn.cluster import KMeans
from ultralytics import YOLO
from google.colab import drive

print("✅ Importaciones correctas")

### Celda 3 — Conexión a Google Drive y rutas
> ⚠️ Modifica `RAIZ` si tu carpeta tiene otro nombre.


In [ ]:
# Celda 3 — Google Drive y rutas del proyecto
drive.mount('/content/drive', force_remount=True)

RAIZ          = '/content/drive/MyDrive/TFG_ALEJANDRO_MARQUEZ'
VIDEOS_DIR    = os.path.join(RAIZ, 'videos_pruebas')
MODELOS_DIR   = os.path.join(RAIZ, 'modelos')
DATOS_DIR     = os.path.join(RAIZ, 'datos')
RESULTADOS_DIR = os.path.join(RAIZ, 'resultados')

for carpeta in [MODELOS_DIR, DATOS_DIR, RESULTADOS_DIR]:
    os.makedirs(carpeta, exist_ok=True)

# Vídeo principal
NOMBRE_VIDEO = 'voley_fondo_01.mp4'   # ← modifica si es necesario
VIDEO_PATH   = os.path.join(VIDEOS_DIR, NOMBRE_VIDEO)
assert os.path.exists(VIDEO_PATH), f"❌ Vídeo no encontrado: {VIDEO_PATH}"

cap = cv2.VideoCapture(VIDEO_PATH)
fps          = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
ancho        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto         = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"🎬 {NOMBRE_VIDEO}")
print(f"   {ancho}×{alto}px | {fps:.1f} fps | {total_frames} frames ({total_frames/fps:.1f}s)")

## Módulo 1 — Calibración de homografía

Mapeamos coordenadas de píxel a metros reales usando 4 puntos de calibración
de la pista de voleibol (9×18 m).

| Punto | Píxel (x, y) | Real (x, y) en metros |
|-------|-------------|----------------------|
| P2 — fondo derecha cerca | (555, 308) | (9, 0) |
| P3 — poste red izquierdo | (103, 175) | (0, 9) |
| P5 — fondo lejos izquierda | (175, 95) | (0, 18) |
| P6 — fondo lejos derecha | (468, 95) | (9, 18) |


### Celda 4 — Cálculo de la homografía


In [ ]:
# Celda 4 — Homografía: píxel → metros reales
# Con 4 puntos usamos getPerspectiveTransform (solución exacta, error=0)

pts_imagen = np.float32([
    [555, 308],   # P2 — fondo derecha cerca
    [103, 175],   # P3 — poste red izquierdo
    [175,  95],   # P5 — fondo lejos izquierda
    [468,  95],   # P6 — fondo lejos derecha
])

pts_reales = np.float32([
    [9,  0],   # P2
    [0,  9],   # P3
    [0, 18],   # P5
    [9, 18],   # P6
])

H     = cv2.getPerspectiveTransform(pts_imagen, pts_reales)
H_inv = np.linalg.inv(H)

def pixel_a_metros(px, py, H):
    punto = np.array([[[px, py]]], dtype=np.float32)
    res   = cv2.perspectiveTransform(punto, H)
    return round(float(res[0][0][0]), 2), round(float(res[0][0][1]), 2)

def metros_a_pixel(xm, ym, H_inv):
    punto = np.array([[[xm, ym]]], dtype=np.float32)
    res   = cv2.perspectiveTransform(punto, H_inv)
    return int(res[0][0][0]), int(res[0][0][1])

# Verificación — error debe ser exactamente 0
print("✅ Homografía calculada. Verificación (error=0.00 en los 4 puntos):")
checks = [
    ("P2 fondo der cerca",   555, 308, 9,  0),
    ("P3 red izq",           103, 175, 0,  9),
    ("P5 fondo lejos izq",   175,  95, 0, 18),
    ("P6 fondo lejos der",   468,  95, 9, 18),
]
for nombre, px, py, xe, ye in checks:
    xm, ym = pixel_a_metros(px, py, H)
    print(f"  {nombre:25s} → ({xm:.2f}m, {ym:.2f}m)  error={abs(xm-xe)+abs(ym-ye):.3f}m")

np.save(os.path.join(DATOS_DIR, 'homografia_H.npy'), H)
print("\n💾 Guardado: datos/homografia_H.npy")

### Celda 5 — Verificación visual de la homografía
> La rejilla cyan debe coincidir con las líneas blancas de la pista. La línea roja = red (y=9m).


In [ ]:
# Celda 5 — Verificación visual: rejilla métrica sobre el frame
cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame = cap.read()
cap.release()
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(frame_rgb)

for x_m in np.arange(0, 9.1, 1.0):
    pts = [(metros_a_pixel(x_m, y_m, H_inv)) for y_m in np.arange(0, 18.1, 0.5)
           if 0 <= metros_a_pixel(x_m, y_m, H_inv)[0] <= ancho
           and 0 <= metros_a_pixel(x_m, y_m, H_inv)[1] <= alto]
    if len(pts) > 1:
        ax.plot([p[0] for p in pts], [p[1] for p in pts], '-', color='cyan', alpha=0.6, linewidth=0.8)

for y_m in np.arange(0, 18.1, 1.0):
    pts = [(metros_a_pixel(x_m, y_m, H_inv)) for x_m in np.arange(0, 9.1, 0.5)
           if 0 <= metros_a_pixel(x_m, y_m, H_inv)[0] <= ancho
           and 0 <= metros_a_pixel(x_m, y_m, H_inv)[1] <= alto]
    if len(pts) > 1:
        color = 'red' if y_m == 9.0 else 'cyan'
        lw    = 1.5   if y_m == 9.0 else 0.8
        ax.plot([p[0] for p in pts], [p[1] for p in pts], '-', color=color, alpha=0.7, linewidth=lw)

ax.set_title("Rejilla métrica — las líneas CYAN deben coincidir con la pista\nLínea ROJA = red (y=9m)", fontsize=10)
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(RESULTADOS_DIR, 'verificacion_homografia.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Guardado: resultados/verificacion_homografia.png")

## Módulo 2 — Detección y tracking

Pipeline: **YOLOv8s** (detección de personas, clase 0) + **ByteTrack** con configuración
optimizada para voleibol (track_buffer=90 = 3 segundos de memoria de ID).

Los pies del jugador (punto inferior del bounding box) se usan como punto
de proyección — están en el suelo, por lo que la homografía los mapea correctamente.


### Celda 6 — Configuración de ByteTrack y extracción de tracking


In [ ]:
# Celda 6 — Configuración ByteTrack + extracción frame a frame
# Configuración optimizada para voleibol:
#   track_buffer=90  → mantiene ID 3 segundos sin detección (evita pérdida en oclusiones)
#   track_high_thresh=0.25 → detecta más jugadores (menos falsos negativos)
#   match_thresh=0.85 → más estricto al asociar IDs (menos ID switches)

config_bytetrack = """tracker_type: bytetrack
track_high_thresh: 0.25
track_low_thresh: 0.05
new_track_thresh: 0.25
track_buffer: 90
match_thresh: 0.85
fuse_score: True
"""
config_path = '/content/bytetrack_custom.yaml'
with open(config_path, 'w') as f:
    f.write(config_bytetrack)

modelo = YOLO('yolov8s.pt')

cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
datos = []
frame_id = 0

print(f"Procesando {total_frames} frames (~2-3 min en Colab con GPU)...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = modelo.track(frame, persist=True, tracker=config_path,
                           classes=[0], conf=0.25, iou=0.45, verbose=False)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes
        for i in range(len(boxes)):
            id_j = int(boxes.id[i])
            x1, y1, x2, y2 = boxes.xyxy[i].tolist()
            conf = float(boxes.conf[i])
            cx_px = (x1 + x2) / 2
            cy_px = y2  # pies del jugador (punto en el suelo)
            x_m, y_m = pixel_a_metros(cx_px, cy_px, H)
            datos.append({'frame': frame_id, 'id': id_j,
                          'x1': round(x1,1), 'y1': round(y1,1),
                          'x2': round(x2,1), 'y2': round(y2,1),
                          'cx_px': round(cx_px,1), 'cy_px': round(cy_px,1),
                          'x_m': x_m, 'y_m': y_m, 'conf': round(conf,3)})

    if frame_id % 300 == 0:
        print(f"  Frame {frame_id}/{total_frames} ({100*frame_id/total_frames:.0f}%)")
    frame_id += 1

cap.release()
df_raw = pd.DataFrame(datos)
df_raw.to_csv(os.path.join(DATOS_DIR, 'tracking_raw.csv'), index=False)
print(f"\n✅ Completado: {len(df_raw)} detecciones | {df_raw['id'].nunique()} IDs únicos")
print("💾 Guardado: datos/tracking_raw.csv")

## Módulo 3 — Limpieza y fusión de IDs

ByteTrack fragmenta los IDs cuando hay oclusiones (un mismo jugador recibe
varios IDs a lo largo del partido). Aplicamos tres estrategias:

1. **Filtrado** por confianza y zona de pista en metros  
2. **Re-identificación** por proximidad espacial (fusión de IDs consecutivos)  
3. **Clustering K-means** (k=6 por mitad de pista) para asignar jugador definitivo


### Celda 7 — Filtrado de detecciones


In [ ]:
# Celda 7 — Filtrado: confianza, zona de pista y presencia mínima
df = pd.read_csv(os.path.join(DATOS_DIR, 'tracking_raw.csv'))
print(f"Detecciones brutas: {len(df)} | IDs únicos: {df['id'].nunique()}")

# Filtro 1: confianza mínima
df = df[df['conf'] >= 0.3]

# Filtro 2: zona de pista en metros (con margen de 0.5m)
df_pista = df[
    (df['x_m'] >= -0.5) & (df['x_m'] <=  9.5) &
    (df['y_m'] >=  0.5) & (df['y_m'] <= 18.5)
].copy()

# Filtro 3: IDs con presencia mínima del 3% de frames (jugadores reales)
total_f  = df_pista['frame'].nunique()
min_f    = int(total_f * 0.03)
conteo   = df_pista.groupby('id')['frame'].nunique()
ids_ok   = conteo[conteo >= min_f].index
df_limpio = df_pista[df_pista['id'].isin(ids_ok)].copy()

print(f"Tras filtros: {len(df_limpio)} detecciones | {df_limpio['id'].nunique()} IDs válidos")
df_limpio.to_csv(os.path.join(DATOS_DIR, 'tracking_v2.csv'), index=False)
print("💾 Guardado: datos/tracking_v2.csv")

### Celda 8 — Fusión de IDs por proximidad espacial
> Si un ID desaparece y otro aparece a <1.5m en <60 frames → probablemente es el mismo jugador.


In [ ]:
# Celda 8 — Re-identificación: fusión de IDs fragmentados por oclusión
df_trabajo = pd.read_csv(os.path.join(DATOS_DIR, 'tracking_v2.csv'))

resumen_ids = df_trabajo.groupby('id').agg(
    frame_ini=('frame','min'), frame_fin=('frame','max'),
    x_ini=('x_m','first'), y_ini=('y_m','first'),
    x_fin=('x_m','last'),  y_fin=('y_m','last'),
    n_frames=('frame','nunique')
).reset_index()

DIST_MAX = 1.5   # metros — máxima distancia espacial para fusionar
GAP_MAX  = 60    # frames — máximo gap temporal (~2 segundos)

fusiones = {}
ids_ord = resumen_ids.sort_values('frame_ini')

for _, id_a in ids_ord.iterrows():
    id_a_real = fusiones.get(id_a['id'], id_a['id'])
    for _, id_b in ids_ord.iterrows():
        if id_a['id'] == id_b['id']:
            continue
        id_b_real = fusiones.get(id_b['id'], id_b['id'])
        if id_a_real == id_b_real:
            continue
        gap  = id_b['frame_ini'] - id_a['frame_fin']
        if not (0 < gap <= GAP_MAX):
            continue
        dist = np.sqrt((id_b['x_ini']-id_a['x_fin'])**2 + (id_b['y_ini']-id_a['y_fin'])**2)
        if dist <= DIST_MAX:
            n_a = resumen_ids[resumen_ids['id']==id_a_real]['n_frames'].values
            n_b = resumen_ids[resumen_ids['id']==id_b_real]['n_frames'].values
            n_a = n_a[0] if len(n_a) > 0 else 0
            n_b = n_b[0] if len(n_b) > 0 else 0
            id_p = id_a_real if n_a >= n_b else id_b_real
            id_s = id_b_real if n_a >= n_b else id_a_real
            fusiones[id_s] = id_p

def resolver_cadena(x, fusiones, max_iter=20):
    actual = x
    for _ in range(max_iter):
        sig = fusiones.get(actual, actual)
        if sig == actual: return actual
        actual = sig
    return actual

df_trabajo['id_fusionado'] = df_trabajo['id'].apply(lambda x: resolver_cadena(x, fusiones))
print(f"IDs antes: {df_trabajo['id'].nunique()} → después de fusión: {df_trabajo['id_fusionado'].nunique()}")
print(f"Fusiones realizadas: {len(fusiones)}")

df_trabajo.to_csv(os.path.join(DATOS_DIR, 'tracking_fusionado.csv'), index=False)
print("💾 Guardado: datos/tracking_fusionado.csv")

### Celda 9 — Clustering K-means (k=6 por mitad de pista)
> En lugar de k=14 global (que mezcla equipos), hacemos k=6 en cada mitad por separado. Garantiza exactamente 6 jugadores por equipo.


In [ ]:
# Celda 9 — Clustering K-means k=6 por mitad de pista
df_trabajo = pd.read_csv(os.path.join(DATOS_DIR, 'tracking_fusionado.csv'))

df_pista = df_trabajo[
    (df_trabajo['x_m'] >= 0) & (df_trabajo['x_m'] <= 9) &
    (df_trabajo['y_m'] >= 0.5) & (df_trabajo['y_m'] <= 18.5) &
    (df_trabajo['conf'] >= 0.35)
].copy()

# Árbitros conocidos (IDs con >80% presencia o dispersión y_std <0.3)
ARBITROS_IDS = [2, 1602]
df_pista = df_pista[~df_pista['id_fusionado'].isin(ARBITROS_IDS)].copy()

df_cerca  = df_pista[df_pista['y_m'] <  9].copy()
df_lejano = df_pista[df_pista['y_m'] >= 9].copy()

k = 6
km_cerca  = KMeans(n_clusters=k, random_state=42, n_init=15)
km_lejano = KMeans(n_clusters=k, random_state=42, n_init=15)

df_cerca['jugador']  = km_cerca.fit_predict(df_cerca[['x_m', 'y_m']])
df_lejano['jugador'] = km_lejano.fit_predict(df_lejano[['x_m', 'y_m']]) + 6

df_final2 = pd.concat([df_cerca, df_lejano], ignore_index=True)

total_f = df_pista['frame'].nunique()
print("Equipo CERCA (J0-J5):")
for j in range(6):
    df_j = df_final2[df_final2['jugador']==j]
    print(f"  J{j}: ({df_j['x_m'].mean():.1f}m, {df_j['y_m'].mean():.1f}m)  {100*df_j['frame'].nunique()/total_f:.0f}% presencia")
print("\nEquipo LEJANO (J6-J11):")
for j in range(6, 12):
    df_j = df_final2[df_final2['jugador']==j]
    print(f"  J{j}: ({df_j['x_m'].mean():.1f}m, {df_j['y_m'].mean():.1f}m)  {100*df_j['frame'].nunique()/total_f:.0f}% presencia")

df_final2.to_csv(os.path.join(DATOS_DIR, 'tracking_final_v2.csv'), index=False)
print("\n💾 Guardado: datos/tracking_final_v2.csv")

## Módulo 4 — Análisis espacial: métricas y dashboard


### Celda 10 — Métricas por jugador


In [ ]:
# Celda 10 — Métricas por jugador (distancia, velocidad, área de cobertura)
df_final2 = pd.read_csv(os.path.join(DATOS_DIR, 'tracking_final_v2.csv'))
total_f   = df_final2['frame'].nunique()
fps       = 30.0
VEL_MAX   = 7.0  # m/s — velocidad máxima humana en voleibol

metricas = []
for j in range(12):
    df_j   = df_final2[df_final2['jugador']==j].sort_values('frame').copy()
    equipo = 'CERCA' if j < 6 else 'LEJANO'
    x_med  = df_j['x_m'].mean()
    y_med  = df_j['y_m'].mean()

    df_j['dx']     = df_j['x_m'].diff()
    df_j['dy']     = df_j['y_m'].diff()
    df_j['dframe'] = df_j['frame'].diff()
    df_j['dist']   = np.sqrt(df_j['dx']**2 + df_j['dy']**2)
    df_j['vel']    = df_j['dist'] / (df_j['dframe'] / fps)

    mask      = (df_j['dframe'] == 1) & (df_j['vel'] <= VEL_MAX)
    distancia = df_j.loc[mask, 'dist'].sum()
    tiempo_s  = mask.sum() / fps
    velocidad = distancia / tiempo_s if tiempo_s > 0 else 0

    puntos = df_j[(df_j['x_m']>=0)&(df_j['x_m']<=9)&
                  (df_j['y_m']>=0)&(df_j['y_m']<=18)][['x_m','y_m']].drop_duplicates().values
    try:    area = round(ConvexHull(puntos).volume, 1)
    except: area = 0.0

    pct = round(100 * df_j['frame'].nunique() / total_f, 1)
    metricas.append({'jugador': j, 'equipo': equipo, 'x_med': round(x_med,2),
                     'y_med': round(y_med,2), 'distancia_m': round(distancia,1),
                     'velocidad_ms': round(velocidad,2), 'area_m2': area, 'pct': pct})

df_met = pd.DataFrame(metricas)
print(df_met[['jugador','equipo','distancia_m','velocidad_ms','area_m2','pct']].to_string(index=False))

df_met.to_csv(os.path.join(DATOS_DIR, 'metricas_finales.csv'), index=False)
print("\n💾 Guardado: datos/metricas_finales.csv")

### Celda 11 — Dashboard definitivo (posiciones, heatmaps, métricas)


In [ ]:
# Celda 11 — Dashboard top-down definitivo
df_final2 = pd.read_csv(os.path.join(DATOS_DIR, 'tracking_final_v2.csv'))
df_met    = pd.read_csv(os.path.join(DATOS_DIR, 'metricas_finales.csv'))

COLOR_CERCA  = '#4FC3F7'
COLOR_LEJANO = '#FF8A65'

fig = plt.figure(figsize=(22, 13))
fig.patch.set_facecolor('#1a1a2e')

ax_main  = fig.add_axes([0.03, 0.08, 0.30, 0.85])
ax_hc    = fig.add_axes([0.36, 0.52, 0.22, 0.40])
ax_hl    = fig.add_axes([0.36, 0.08, 0.22, 0.40])
ax_stats = fig.add_axes([0.61, 0.08, 0.37, 0.85])

def dibujar_pista_base(ax):
    ax.add_patch(plt.Rectangle((0,0), 9, 18, facecolor='#8B7355', edgecolor='white', linewidth=2))
    ax.axhline(y=9,  color='white', linewidth=2.5, zorder=3)
    ax.axhline(y=3,  color='white', linewidth=1, linestyle='--', alpha=0.5, zorder=3)
    ax.axhline(y=15, color='white', linewidth=1, linestyle='--', alpha=0.5, zorder=3)
    ax.set_xlim(-0.5, 9.5); ax.set_ylim(-0.5, 19)
    ax.set_facecolor('#1a1a2e'); ax.tick_params(colors='white', labelsize=8)

dibujar_pista_base(ax_main)
for j in range(12):
    df_j  = df_final2[df_final2['jugador']==j]
    color = COLOR_CERCA if j < 6 else COLOR_LEJANO
    x_med = df_j['x_m'].mean(); y_med = df_j['y_m'].mean()
    muestra = df_j.sample(min(300, len(df_j)))
    ax_main.plot(muestra['x_m'], muestra['y_m'], '.', color=color, alpha=0.07, markersize=1.5)
    ax_main.plot(x_med, y_med, 'o', color=color, markersize=15, markeredgecolor='white', markeredgewidth=2, zorder=5)
    ax_main.text(x_med, y_med, f'J{j}', ha='center', va='center', fontsize=7, fontweight='bold', color='white', zorder=6)
ax_main.text(4.5, 9.35, 'RED', ha='center', color='white', fontsize=10, fontweight='bold')
ax_main.text(4.5, 18.7, 'Equipo LEJANO', ha='center', color=COLOR_LEJANO, fontsize=10, fontweight='bold')
ax_main.text(4.5, -0.4, 'Equipo CERCA', ha='center', color=COLOR_CERCA, fontsize=10, fontweight='bold')
ax_main.set_xlabel("Ancho (m)", color='white', fontsize=9); ax_main.set_ylabel("Largo (m)", color='white', fontsize=9)
ax_main.set_title("Posiciones medias", color='white', fontsize=11, pad=8)

def heatmap(ax, df_eq, cmap, titulo):
    dibujar_pista_base(ax)
    x = df_eq['x_m'].values; y = df_eq['y_m'].values
    mask = (x>=0)&(x<=9)&(y>=0)&(y<=18); x, y = x[mask], y[mask]
    if len(x) > 20:
        kde = gaussian_kde(np.vstack([x, y]), bw_method=0.18)
        xi  = np.linspace(0, 9, 60); yi = np.linspace(0, 18, 120)
        Xi, Yi = np.meshgrid(xi, yi)
        Zi = kde(np.vstack([Xi.ravel(), Yi.ravel()])).reshape(Xi.shape)
        ax.contourf(Xi, Yi, Zi, levels=14, cmap=cmap, alpha=0.8, zorder=2)
    ax.set_title(titulo, color='white', fontsize=10, pad=5)
    ax.set_xlabel("Ancho (m)", color='white', fontsize=8)

heatmap(ax_hc, df_final2[df_final2['jugador'] < 6],  'Blues',   'Heatmap equipo CERCA')
heatmap(ax_hl, df_final2[df_final2['jugador'] >= 6], 'Oranges', 'Heatmap equipo LEJANO')

ax_stats.set_facecolor('#1a1a2e'); ax_stats.axis('off')
ax_stats.set_title("Métricas por jugador", color='white', fontsize=12, pad=10, fontweight='bold')
cols  = ['Jugador', 'Equipo', 'Dist (m)', 'Vel (m/s)', 'Area (m²)', 'Presencia']
col_x = [0.01, 0.18, 0.38, 0.56, 0.73, 0.88]
y = 0.95
for cx, c in zip(col_x, cols):
    ax_stats.text(cx, y, c, color='#FFD700', fontsize=9.5, fontweight='bold', transform=ax_stats.transAxes)
y -= 0.03
ax_stats.plot([0,1],[y,y], color='gray', linewidth=0.5, transform=ax_stats.transAxes)
for _, row in df_met.sort_values(['equipo','y_med']).iterrows():
    y -= 0.07
    color = COLOR_CERCA if row['equipo']=='CERCA' else COLOR_LEJANO
    for cx, v in zip(col_x, [f"J{int(row['jugador'])}", row['equipo'],
                              f"{row['distancia_m']:.0f}", f"{row['velocidad_ms']:.2f}",
                              f"{row['area_m2']:.0f}", f"{row['pct']:.0f}%"]):
        ax_stats.text(cx, y, v, color=color, fontsize=9, transform=ax_stats.transAxes, va='center')

fig.text(0.5, 0.99, "TFG — Análisis automático de jugadas de voleibol\nYOLOv8s + ByteTrack + Homografía + K-means (k=6/equipo)",
         ha='center', va='top', color='white', fontsize=13, fontweight='bold')

plt.savefig(os.path.join(RESULTADOS_DIR, 'dashboard_definitivo.png'), dpi=180, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print("✅ Guardado: resultados/dashboard_definitivo.png")

## Módulo 5 — Carga física

Calculamos métricas de carga física a partir de las trayectorias, siguiendo
las zonas de intensidad estándar en medicina deportiva (equivalente GPS desde vídeo):

| Zona | Rango velocidad | Interpretación |
|------|----------------|----------------|
| Z1 | 0-1 m/s | Parado / andando |
| Z2 | 1-2 m/s | Trote |
| Z3 | 2-3.5 m/s | Carrera media |
| Z4 | 3.5-5.5 m/s | Carrera rápida |
| Z5 | >5.5 m/s | Sprint |


### Celda 12 — Cálculo de carga física por zonas


In [ ]:
# Celda 12 — Carga física: zonas de intensidad por jugador
df_final2 = pd.read_csv(os.path.join(DATOS_DIR, 'tracking_final_v2.csv'))
fps     = 30.0
VEL_MAX = 7.0

ZONAS = [
    (0.0, 1.0, 'Z1 Parado/andando',  '#2196F3'),
    (1.0, 2.0, 'Z2 Trote',           '#4CAF50'),
    (2.0, 3.5, 'Z3 Carrera media',   '#FFC107'),
    (3.5, 5.5, 'Z4 Carrera rapida',  '#FF5722'),
    (5.5, 9.9, 'Z5 Sprint',          '#E91E63'),
]

carga_total = []
for j in range(12):
    df_j   = df_final2[df_final2['jugador']==j].sort_values('frame').copy()
    equipo = 'CERCA' if j < 6 else 'LEJANO'
    df_j['dx']     = df_j['x_m'].diff()
    df_j['dy']     = df_j['y_m'].diff()
    df_j['dframe'] = df_j['frame'].diff()
    df_j['dist']   = np.sqrt(df_j['dx']**2 + df_j['dy']**2)
    df_j['vel_ms'] = df_j['dist'] / (df_j['dframe'] / fps)
    df_j['acel']   = df_j['vel_ms'].diff() / (df_j['dframe'] / fps)

    mask      = (df_j['dframe'] == 1) & (df_j['vel_ms'] <= VEL_MAX)
    df_valido = df_j[mask].copy()
    if len(df_valido) < 10:
        continue

    tiempo_total_s = len(df_valido) / fps
    zona_data = {}
    for v_min, v_max, nombre, color in ZONAS:
        en_zona = df_valido[(df_valido['vel_ms']>=v_min)&(df_valido['vel_ms']<v_max)]
        t_zona  = len(en_zona) / fps
        pct     = 100 * t_zona / tiempo_total_s if tiempo_total_s > 0 else 0
        zona_data[nombre] = {'tiempo_s': round(t_zona,1), 'dist_m': round(en_zona['dist'].sum(),1), 'pct': round(pct,1), 'color': color}

    player_load = round(float(np.sqrt((df_valido['dx']**2 + df_valido['dy']**2)).sum()), 1)
    acel_max    = round(float(df_valido['acel'].dropna().abs().quantile(0.95)), 2)
    pct_alta    = round(100 * (df_valido['vel_ms'] > 3.5).sum() / len(df_valido), 1)

    carga_total.append({'jugador': j, 'equipo': equipo, 'tiempo_s': round(tiempo_total_s,1),
                        'dist_total': round(df_valido['dist'].sum(),1),
                        'vel_max': round(float(df_valido['vel_ms'].quantile(0.95)),2),
                        'acel_max': acel_max, 'pct_alta_intensidad': pct_alta,
                        'player_load': player_load, 'zonas': zona_data})

df_carga = pd.DataFrame([{k:v for k,v in c.items() if k != 'zonas'} for c in carga_total])
df_carga.to_csv(os.path.join(DATOS_DIR, 'carga_fisica.csv'), index=False)
print("✅ Carga física calculada para 12 jugadores")
print(df_carga[['jugador','equipo','dist_total','vel_max','pct_alta_intensidad','player_load']].to_string(index=False))
print("\n💾 Guardado: datos/carga_fisica.csv")

### Celda 13 — Visualización de carga física


In [ ]:
# Celda 13 — Visualización: barras apiladas de zonas de intensidad
# (Requiere haber ejecutado la Celda 12 — variable carga_total en memoria)

colores_zonas = {'Z1 Parado/andando':'#1565C0','Z2 Trote':'#2E7D32',
                 'Z3 Carrera media':'#F9A825','Z4 Carrera rapida':'#E64A19','Z5 Sprint':'#AD1457'}

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
fig.patch.set_facecolor('#1a1a2e')

for ax_idx, (equipo, j_rng, titulo) in enumerate([
    ('CERCA',  range(0,6),  'Equipo CERCA'),
    ('LEJANO', range(6,12), 'Equipo LEJANO'),
]):
    ax  = axes[ax_idx]
    ax.set_facecolor('#1a1a2e')
    jug = [c for c in carga_total if c['jugador'] in j_rng]
    ypos = np.arange(len(jug))
    acum = np.zeros(len(jug))
    for zona, color in colores_zonas.items():
        vals = [c['zonas'].get(zona, {}).get('pct', 0) for c in jug]
        ax.barh(ypos, vals, left=acum, color=color, alpha=0.85,
                label=zona if ax_idx == 0 else '')
        for i,(v,a) in enumerate(zip(vals,acum)):
            if v > 5:
                ax.text(a+v/2, i, f'{v:.0f}%', ha='center', va='center', color='white', fontsize=7.5, fontweight='bold')
        acum += np.array(vals)
    ax.set_yticks(ypos); ax.set_yticklabels([f"J{c['jugador']}" for c in jug], color='white', fontsize=11)
    ax.set_xlim(0, 100); ax.set_xlabel("% tiempo en zona", color='white', fontsize=10)
    ax.set_title(titulo, color='white', fontsize=13, fontweight='bold', pad=10)
    ax.tick_params(colors='white')
    for sp in ['top','right']: ax.spines[sp].set_visible(False)
    for sp in ['bottom','left']: ax.spines[sp].set_color('gray')

handles = [plt.Rectangle((0,0),1,1,color=c,alpha=0.85) for c in colores_zonas.values()]
fig.legend(handles, colores_zonas.keys(), loc='lower center', ncol=5,
           bbox_to_anchor=(0.5,-0.02), facecolor='#2a2a3e', edgecolor='gray', labelcolor='white', fontsize=9)

fig.text(0.5, 1.01, "Análisis de carga física — Distribución por zonas de intensidad\nMetodología: zonas de velocidad estándar en medicina deportiva",
         ha='center', va='top', color='white', fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(os.path.join(RESULTADOS_DIR, 'carga_fisica_zonas.png'), dpi=180, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print("✅ Guardado: resultados/carga_fisica_zonas.png")

## Módulo 6 — Estimación de pose y clasificación de acciones

Usamos **YOLOv8s-pose** para detectar 17 keypoints del cuerpo humano en cada jugador.
A partir de los ángulos articulares aplicamos un clasificador basado en reglas (*rule-based*)
para identificar acciones técnicas de voleibol.

**Keypoints COCO:** 0=nariz, 5-6=hombros, 7-10=codos/muñecas, 11-12=caderas, 13-16=rodillas/tobillos

**Reglas de clasificación:**
| Acción | Condición |
|--------|-----------|
| REMATE_BLOQUEO | Brazos arriba + rodilla > 150° |
| SALTO_ACCION | Brazos arriba + rodilla < 150° |
| RECEPCION | Rodilla < 130° |
| POSICION_BASE | 130° ≤ rodilla < 160° |
| EN_PIE | Rodilla ≥ 160° |


### Celda 14 — Definición de constantes y funciones de pose


In [ ]:
# Celda 14 — Constantes y funciones para análisis de pose

# Conexiones del esqueleto (pares de keypoints COCO)
ESQUELETO = [
    (0,1),(0,2),(1,3),(2,4),          # cabeza
    (5,6),(5,7),(7,9),(6,8),(8,10),   # brazos
    (5,11),(6,12),(11,12),            # tronco
    (11,13),(13,15),(12,14),(14,16),  # piernas
]

COLORES_ACCION = {
    'REMATE_BLOQUEO': '#FF4444',
    'SALTO_ACCION'  : '#FF8800',
    'RECEPCION'     : '#44FF44',
    'POSICION_BASE' : '#4488FF',
    'EN_PIE'        : '#AAAAAA',
    'INDETERMINADO' : '#666666',
}

def _angulo(p1, p2, p3, c1, c2, c3, umbral=0.3):
    """Ángulo en p2 formado por p1-p2-p3. Devuelve None si confianza insuficiente."""
    if min(c1, c2, c3) < umbral:
        return None
    v1 = p1 - p2; v2 = p3 - p2
    cos_a = np.clip(np.dot(v1,v2)/(np.linalg.norm(v1)*np.linalg.norm(v2)+1e-6), -1, 1)
    return float(np.degrees(np.arccos(cos_a)))

def clasificar_accion(kps, confs):
    """
    Clasifica la acción de un jugador a partir de sus 17 keypoints y sus confianzas.
    kps: array (17, 2), confs: array (17,)
    """
    brazos_arriba = False
    if confs[9] > 0.3 and confs[5] > 0.3:
        brazos_arriba = kps[9][1] < kps[5][1]
    if confs[10] > 0.3 and confs[6] > 0.3:
        brazos_arriba = brazos_arriba or (kps[10][1] < kps[6][1])

    rod_vals = []
    r_izq = _angulo(kps[11], kps[13], kps[15], confs[11], confs[13], confs[15])
    r_der = _angulo(kps[12], kps[14], kps[16], confs[12], confs[14], confs[16])
    if r_izq is not None: rod_vals.append(r_izq)
    if r_der is not None: rod_vals.append(r_der)
    rod_med = np.mean(rod_vals) if rod_vals else None

    if brazos_arriba:
        return 'SALTO_ACCION' if (rod_med is not None and rod_med < 150) else 'REMATE_BLOQUEO'
    else:
        if rod_med is None:        return 'INDETERMINADO'
        elif rod_med < 130:        return 'RECEPCION'
        elif rod_med < 160:        return 'POSICION_BASE'
        else:                      return 'EN_PIE'

print("✅ Constantes y funciones de pose definidas")

### Celda 15 — Análisis de pose: extracción de ángulos articulares
> Se analiza 1 de cada 15 frames (~2fps) para cubrir todo el vídeo sin procesar los ~3000 frames completos.


In [ ]:
# Celda 15 — Análisis de pose en frames muestreados
modelo_pose  = YOLO('yolov8s-pose.pt')
cap          = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
PASO         = 15  # 1 de cada 15 frames ≈ 2fps

frames_pose = []
frame_id = 0; procesados = 0
print(f"Analizando pose cada {PASO} frames ({total_frames//PASO} frames totales)...")

while True:
    ret, frame = cap.read()
    if not ret: break
    if frame_id % PASO == 0:
        results = modelo_pose(frame, conf=0.4, verbose=False)
        if results[0].keypoints is not None:
            kpts_data = results[0].keypoints.xy.cpu().numpy()
            conf_data = results[0].keypoints.conf.cpu().numpy()
            boxes     = results[0].boxes.xyxy.cpu().numpy()
            for i in range(len(kpts_data)):
                x1,y1,x2,y2 = boxes[i]
                xm,ym = pixel_a_metros((x1+x2)/2, y2, H)
                if not (0 <= xm <= 9 and 0.5 <= ym <= 18.5):
                    continue
                kp = kpts_data[i]; cf = conf_data[i]
                r_izq = _angulo(kp[11],kp[13],kp[15],cf[11],cf[13],cf[15])
                r_der = _angulo(kp[12],kp[14],kp[16],cf[12],cf[14],cf[16])
                h_izq = _angulo(kp[11],kp[5],kp[7],  cf[11],cf[5],cf[7])
                h_der = _angulo(kp[12],kp[6],kp[8],  cf[12],cf[6],cf[8])
                brazos_arriba = False
                if cf[9]>0.3 and cf[5]>0.3: brazos_arriba = kp[9][1] < kp[5][1]
                if cf[10]>0.3 and cf[6]>0.3: brazos_arriba = brazos_arriba or (kp[10][1] < kp[6][1])
                frames_pose.append({'frame': frame_id, 'x_m': round(xm,2), 'y_m': round(ym,2),
                    'ang_rodilla_izq': round(r_izq,1) if r_izq else None,
                    'ang_rodilla_der': round(r_der,1) if r_der else None,
                    'ang_hombro_izq':  round(h_izq,1) if h_izq else None,
                    'ang_hombro_der':  round(h_der,1) if h_der else None,
                    'brazos_arriba': brazos_arriba, 'equipo': 'LEJANO' if ym>9 else 'CERCA'})
        procesados += 1
        if procesados % 50 == 0: print(f"  Frame {frame_id}/{total_frames} ({100*frame_id/total_frames:.0f}%)")
    frame_id += 1

cap.release()
df_pose = pd.DataFrame(frames_pose)
df_pose.to_csv(os.path.join(DATOS_DIR, 'pose_analisis.csv'), index=False)
print(f"\n✅ {len(df_pose)} detecciones de pose en {procesados} frames analizados")
print("💾 Guardado: datos/pose_analisis.csv")

### Celda 16 — Clasificación de acciones y visualización


In [ ]:
# Celda 16 — Clasificación de acciones + visualización en frame
df_pose = pd.read_csv(os.path.join(DATOS_DIR, 'pose_analisis.csv'))

# ── Clasificar acciones ─────────────────────────────────────
def _clasificar_row(row):
    kps_dummy = np.zeros((17,2))
    confs_dummy = np.zeros(17)
    brazos = row['brazos_arriba']
    rod_vals = [v for v in [row.get('ang_rodilla_izq'), row.get('ang_rodilla_der')] if pd.notna(v)]
    rod_med = np.mean(rod_vals) if rod_vals else None
    if brazos:
        return 'SALTO_ACCION' if (rod_med is not None and rod_med < 150) else 'REMATE_BLOQUEO'
    else:
        if rod_med is None:   return 'INDETERMINADO'
        elif rod_med < 130:   return 'RECEPCION'
        elif rod_med < 160:   return 'POSICION_BASE'
        else:                 return 'EN_PIE'

df_pose['accion'] = df_pose.apply(_clasificar_row, axis=1)
df_pose.to_csv(os.path.join(DATOS_DIR, 'pose_acciones.csv'), index=False)

print("📊 Distribución de acciones:")
total = len(df_pose)
for accion, n in df_pose['accion'].value_counts().items():
    print(f"  {accion:20s}: {'█'*int(100*n/total/2):<40s} {n:4d} ({100*n/total:.1f}%)")

# ── Visualización en frame 1425 ──────────────────────────────
FRAME_OBJETIVO = 1425
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_OBJETIVO)
ret, frame = cap.read()
cap.release()

frame_rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
results_pose  = modelo_pose(frame_rgb, conf=0.35, verbose=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 8), dpi=120)
fig.patch.set_facecolor('#0f0f1a')
ax0, ax1 = axes
ax0.imshow(frame_rgb); ax0.set_title(f'Pose — Frame {FRAME_OBJETIVO}', fontsize=13, fontweight='bold'); ax0.axis('off')
ax1.set_facecolor('#1a1a2e'); ax1.axis('off'); ax1.set_title(f'Acciones detectadas', fontsize=13, fontweight='bold', color='white')

personas = []
if results_pose[0].keypoints is not None:
    kpts_all = results_pose[0].keypoints.xy.cpu().numpy()
    conf_all = results_pose[0].keypoints.conf.cpu().numpy()
    boxes_all = results_pose[0].boxes.xyxy.cpu().numpy()
    for i in range(len(kpts_all)):
        x1,y1,x2,y2 = boxes_all[i]
        xm,ym = pixel_a_metros((x1+x2)/2, y2, H)
        if not (0 <= xm <= 9 and 0.5 <= ym <= 18.5): continue
        kp = kpts_all[i]; cf = conf_all[i]
        accion = clasificar_accion(kp, cf)
        color  = COLORES_ACCION.get(accion, 'white')
        personas.append((i, accion))
        for p1i,p2i in ESQUELETO:
            if cf[p1i]>0.3 and cf[p2i]>0.3:
                ax0.plot([kp[p1i][0],kp[p2i][0]], [kp[p1i][1],kp[p2i][1]], '-', color=color, linewidth=2, alpha=0.85)
        for ki in range(17):
            if cf[ki]>0.3: ax0.plot(kp[ki][0], kp[ki][1], 'o', color=color, markersize=5, markeredgecolor='white', markeredgewidth=0.8)
        if cf[0]>0.3:
            ax0.text(kp[0][0], kp[0][1]-15, accion, color='white', fontsize=7, fontweight='bold', ha='center',
                     bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.75, edgecolor='none'))

import matplotlib.patches as mpatches
ax0.legend(handles=[mpatches.Patch(color=v, label=k) for k,v in COLORES_ACCION.items()],
           loc='upper right', fontsize=7, framealpha=0.7)

y_pos = 0.92
ax1.text(0.5, y_pos, f'Personas detectadas: {len(personas)}', transform=ax1.transAxes,
         ha='center', fontsize=11, color='#00d4ff', fontweight='bold')
for pid, accion in personas:
    y_pos -= 0.08
    ax1.text(0.15, y_pos, f'Jugador {pid+1}:', transform=ax1.transAxes, fontsize=10, color='#aaaaaa')
    ax1.text(0.55, y_pos, accion, transform=ax1.transAxes, fontsize=10,
             color=COLORES_ACCION.get(accion,'white'), fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(RESULTADOS_DIR, 'pose_frame1425.png'), dpi=120, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print("✅ Guardado: resultados/pose_frame1425.png")
print("💾 Guardado: datos/pose_acciones.csv")

## Módulo 7 — Evaluación del tracker

Métricas cuantitativas del sistema. En un trabajo de investigación con ground truth
anotado manualmente obtendríamos MOTA/MOTP. Aquí usamos evaluación semi-automática:
contamos detecciones por frame comparadas con los 12 jugadores reales esperados.


### Celda 17 — Métricas de evaluación


In [ ]:
# Celda 17 — Evaluación del sistema de tracking
df_v2 = pd.read_csv(os.path.join(DATOS_DIR, 'tracking_v2.csv'))
fps   = 30.0
total_frames  = df_v2['frame'].nunique()
N_JUGADORES_R = 12

df_pista = df_v2[
    (df_v2['x_m']>=0)&(df_v2['x_m']<=9)&
    (df_v2['y_m']>=0.5)&(df_v2['y_m']<=18.5)&
    (df_v2['conf']>=0.35)
].copy()

det_por_frame     = df_pista.groupby('frame')['id'].nunique()
recall_medio      = (det_por_frame.clip(upper=N_JUGADORES_R)/N_JUGADORES_R).mean()
frames_completos  = (det_por_frame >= N_JUGADORES_R).sum()
n_ids_total       = df_pista['id'].nunique()
fragmentacion     = n_ids_total / N_JUGADORES_R
conf_media        = df_pista['conf'].mean()

# ID Switches: cambios de ID en posiciones cercanas entre frames consecutivos
frames_lista = sorted(df_pista['frame'].unique())
id_switches  = 0
for i in range(1, len(frames_lista)):
    f_prev, f_curr = frames_lista[i-1], frames_lista[i]
    if f_curr - f_prev > 3: continue
    d_prev = df_pista[df_pista['frame']==f_prev][['id','x_m','y_m']].values
    d_curr = df_pista[df_pista['frame']==f_curr][['id','x_m','y_m']].values
    for dp in d_prev:
        for dc in d_curr:
            if np.sqrt((dp[1]-dc[1])**2+(dp[2]-dc[2])**2) < 1.0 and dp[0] != dc[0]:
                id_switches += 1; break

print("=" * 55)
print("EVALUACIÓN DEL SISTEMA — TABLA PARA LA MEMORIA")
print("=" * 55)
print(f"  Recall medio (detección)       {recall_medio:.3f}")
print(f"  Frames con detección completa  {100*frames_completos/total_frames:.1f}%")
print(f"  Factor fragmentación IDs       {fragmentacion:.1f}x")
print(f"  ID switches totales            {id_switches}")
print(f"  ID switches/minuto             {id_switches/(total_frames/fps/60):.1f}")
print(f"  Confianza media detección      {conf_media:.3f}")

resumen_eval = {'recall_medio': round(recall_medio,3),
                'pct_deteccion_completa': round(100*frames_completos/total_frames,1),
                'factor_fragmentacion': round(fragmentacion,1),
                'id_switches_total': id_switches,
                'id_switches_por_min': round(id_switches/(total_frames/fps/60),1),
                'confianza_media': round(conf_media,3)}
pd.DataFrame([resumen_eval]).to_csv(os.path.join(DATOS_DIR, 'evaluacion_tracker.csv'), index=False)
print("\n💾 Guardado: datos/evaluacion_tracker.csv")

## Módulo 8 — Exportación de vídeo anotado


### Celda 18 — Generar vídeo con anotaciones y minimapa
> ⚠️ Esta celda tarda ~10-15 minutos. Genera el vídeo completo con bounding boxes, IDs, coordenadas métricas y un minimapa top-down en la esquina.


In [ ]:
# Celda 18 — Vídeo anotado con minimapa top-down
modelo_video = YOLO('yolov8s.pt')

COLOR_CERCA  = (255, 180,  50)
COLOR_LEJANO = ( 50, 140, 255)
COLOR_FUERA  = ( 80,  80,  80)

cap     = cv2.VideoCapture(VIDEO_PATH)
fps_vid = cap.get(cv2.CAP_PROP_FPS)
ancho_v = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto_v  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_v = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

output_path = os.path.join(RESULTADOS_DIR, 'video_anotado.mp4')
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(output_path, fourcc, fps_vid, (ancho_v, alto_v))

MAPA_W=180; MAPA_H=280; MAPA_X=ancho_v-MAPA_W-10; MAPA_Y=10; margen=10

def dibujar_minimapa(frame, detecciones):
    mapa = np.full((MAPA_H, MAPA_W, 3), 30, dtype=np.uint8)
    pw = MAPA_W-2*margen; ph = MAPA_H-2*margen
    cv2.rectangle(mapa,(margen,margen),(margen+pw,margen+ph),(80,120,80),-1)
    cv2.line(mapa,(margen,margen+ph//2),(margen+pw,margen+ph//2),(255,255,255),2)
    for xm,ym,color in detecciones:
        if 0<=xm<=9 and 0<=ym<=18:
            px=margen+int(xm/9*pw); py=margen+int(ym/18*ph)
            cv2.circle(mapa,(px,py),4,color,-1); cv2.circle(mapa,(px,py),4,(255,255,255),1)
    frame[MAPA_Y:MAPA_Y+MAPA_H, MAPA_X:MAPA_X+MAPA_W] = mapa
    cv2.rectangle(frame,(MAPA_X-1,MAPA_Y-1),(MAPA_X+MAPA_W+1,MAPA_Y+MAPA_H+1),(200,200,200),1)
    return frame

print(f"Generando vídeo anotado ({total_v} frames)...")
frame_id = 0
while True:
    ret, frame = cap.read()
    if not ret: break
    fa = frame.copy(); det_mapa = []
    results = modelo_video.track(frame, persist=True, tracker=config_path, classes=[0], conf=0.35, verbose=False)
    if results[0].boxes.id is not None:
        for i in range(len(results[0].boxes)):
            x1,y1,x2,y2 = [int(v) for v in results[0].boxes.xyxy[i].tolist()]
            id_j = int(results[0].boxes.id[i])
            xm,ym = pixel_a_metros((x1+x2)/2, y2, H)
            en = 0<=xm<=9 and 0.5<=ym<=18.5
            c = (COLOR_LEJANO if ym>9 else COLOR_CERCA) if en else COLOR_FUERA
            eq = ('LEJ' if ym>9 else 'CER') if en else ''
            cv2.rectangle(fa,(x1,y1),(x2,y2),c,2)
            lbl = f'ID{id_j} {eq}'
            cv2.rectangle(fa,(x1,y1-16),(x1+len(lbl)*7,y1),c,-1)
            cv2.putText(fa,lbl,(x1+2,y1-3),cv2.FONT_HERSHEY_SIMPLEX,0.4,(255,255,255),1)
            if en:
                cv2.putText(fa,f'{xm:.1f}m,{ym:.1f}m',(x1,y2+12),cv2.FONT_HERSHEY_SIMPLEX,0.35,c,1)
                det_mapa.append((xm,ym,c))
    fa = dibujar_minimapa(fa, det_mapa)
    cv2.putText(fa,f'Frame {frame_id} | {frame_id/fps_vid:.1f}s',(10,alto_v-10),cv2.FONT_HERSHEY_SIMPLEX,0.4,(180,180,180),1)
    cv2.putText(fa,'TFG - Analisis de voleibol | YOLOv8s + ByteTrack + Homografia',(10,20),cv2.FONT_HERSHEY_SIMPLEX,0.45,(220,220,220),1)
    out.write(fa)
    if frame_id % 300 == 0: print(f"  Frame {frame_id}/{total_v} ({100*frame_id/total_v:.0f}%)")
    frame_id += 1

cap.release(); out.release()
print(f"\n✅ Vídeo generado: {output_path}")

---

## Archivos generados

### `datos/`
| Archivo | Descripción |
|---------|-------------|
| `homografia_H.npy` | Matriz de homografía 3×3 |
| `tracking_raw.csv` | Detecciones brutas de YOLO+ByteTrack |
| `tracking_v2.csv` | Filtrado por confianza y zona de pista |
| `tracking_fusionado.csv` | Tras fusión de IDs por proximidad |
| `tracking_final_v2.csv` | Dataset principal con jugador asignado (K-means) |
| `metricas_finales.csv` | Distancia, velocidad, área y presencia por jugador |
| `carga_fisica.csv` | Métricas de carga física y zonas Z1-Z5 |
| `pose_analisis.csv` | Ángulos articulares por frame |
| `pose_acciones.csv` | Clasificación de acciones por jugador |
| `evaluacion_tracker.csv` | Métricas de evaluación del tracker |

### `resultados/`
| Archivo | Descripción |
|---------|-------------|
| `verificacion_homografia.png` | Rejilla métrica sobre el frame |
| `dashboard_definitivo.png` | Dashboard top-down con heatmaps y métricas |
| `carga_fisica_zonas.png` | Distribución de zonas de intensidad |
| `pose_frame1425.png` | Visualización de pose con clasificación de acciones |
| `video_anotado.mp4` | Vídeo de salida con todas las anotaciones |
